In [7]:
# ---------------------------------------------------------------------------
# Imports: Core AutoGen components and utilities
# ---------------------------------------------------------------------------
import autogen
from autogen import AssistantAgent, UserProxyAgent,GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from pathlib import Path
from IPython.display import Image, Markdown
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()  # Loads .env from the current directory (or parent directories)

True

In [4]:
# Step 2a: Load model configuration from CONFIG_LIST.json using the modern LLMConfig API.
# Keep this file out of source control; use .env or env vars for the actual API key.
llm_config_obj = autogen.LLMConfig.from_json(path="CONFIG_LIST.json")
config_list = llm_config_obj.config_list

# Optional: Filter config_list by model name
# Example: Filter to use only "llama-3.3-70b-versatile"
# config_list = [config for config in config_list if config.get("model") == "llama-3.3-70b-versatile"]

# Or filter to use only "openai/gpt-oss-120b"
config_list = [config for config in config_list if config.get("model") == "openai/gpt-oss-120b"]

# Or filter by partial model name match
# config_list = [config for config in config_list if "llama" in config.get("model", "").lower()]

llm_config = {
    "cache_seed": 42,
    "temperature": 0,
    "config_list": config_list,
    "timeout": 120,
}

In [5]:
# Agent 1: Admin (Human Proxy)
# Represents the human user who approves plans and provides feedback.
# code_execution_config=False: This agent doesn't execute code, only approves plans.
user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss\
        the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False
)

# Agent 2: Engineer (Code Writer)
# Writes Python/shell code to scrape arXiv, process data, and create markdown tables.
# Automatically fixes errors based on Executor feedback and iterates until code works.
engineer = AssistantAgent(
    name="Engineer",
    llm_config=llm_config,
    system_message="""Engineer. You follow an approved plan. You write python/shell code to solve tasks. Wrap the code in a code block that specifies the script type. The user can't modify your code. So do not suggest incomplete code which requires others to modify. Don't use a code block if it's not intended to be executed by the executor.
Don't include multiple code blocks in one response. Do not ask others to copy and paste the result. Check the execution result returned by the executor.
If the result indicates there is an error, fix the error and output the code again. Suggest the full code instead of partial code or code changes. If the error can't be fixed or if the task is not solved even after the code is executed successfully, analyze the problem, revisit your assumption, collect additional info you need, and think of a different approach to try.
""",
)

# Agent 3: Scientist (Content Reviewer)
# Reviews paper abstracts and categorizes them by domain (e.g., Education, Healthcare).
# Does not write code — focuses on content analysis and categorization.
scientist = AssistantAgent(
    name="Scientist",
    llm_config=llm_config,
    system_message="""Scientist. You follow an approved plan.
    You are able to categorize papers after seeing their abstracts printed.
    You don't write code.""",
)

# Agent 4: Planner (Workflow Designer)
# Creates the initial plan and revises it based on Admin and Critic feedback.
# Clearly assigns tasks to Engineer (code) vs Scientist (review) until Admin approves.
planner = AssistantAgent(
    name="Planner",
    system_message="""Planner. Suggest a plan. Revise the plan based on feedback
    from admin and critic, until admin approval. The plan may involve an engineer
    who can write code and a scientist who doesn't write code. Explain the plan 
    first. Be clear which step is performed by an engineer, and which step is 
    performed by a scientist.
    """,
    llm_config=llm_config
)

# Agent 5: Executor (Code Runner)
# Automatically executes code written by the Engineer and reports results back.
# human_input_mode="NEVER": runs without prompting for user input
# work_dir="paper": creates a "paper" folder for all generated files and outputs
executor = UserProxyAgent(
    name="Executor",
    system_message="Executor. Execute the code written by the engineer\
        and report the result.",
    human_input_mode="NEVER",
    code_execution_config={"last_n_messages": 3,
                           "work_dir": "paper",
                           "use_docker" : False},
)

# Agent 6: Critic (Reviewer)
# Reviews plans, code, and outputs from other agents to ensure quality.
# Ensures verifiable information (e.g., source URLs) is included in outputs.
critic = AssistantAgent(
    name="Critic", 
    system_message="""Critic. Double check plan, claims, code from other
    agents and provide feedback. Check whether the plan includes adding 
    verifiable info such as source URL.
    """,
    llm_config=llm_config,
)


In [8]:
# ---------------------------------------------------------------------------
# Step 5: Create GroupChat and Manager
# ---------------------------------------------------------------------------
# GroupChat: Container that holds all agents and manages the conversation flow.
# max_round=10: Limits conversation to 10 rounds to prevent infinite loops.
groupchat = GroupChat(
    agents=[user_proxy, engineer, scientist, planner, executor, critic], 
    messages=[], 
    max_round=10
)

# GroupChatManager: Uses an LLM to decide which agent should speak next
# based on the conversation context and each agent's role.
manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)

## Run the research task

**How it works:**
1. The **Admin** sends the research request to the **manager**
2. The **manager** routes messages to the appropriate agents (Planner, Critic, Engineer, Scientist, Executor)
3. The **Planner** creates a plan, the **Critic** reviews it, and the **Admin** approves it
4. The **Engineer** writes code, the **Executor** runs it, and the **Scientist** reviews the results
5. The process continues until the task is complete or max_round is reached

**💡 Tip:** When prompted for plan approval, you can press **Enter** to auto-approve and let the agents continue automatically.

In [9]:
# Initiate the multi-agent conversation
# The Admin (user_proxy) sends the research task to the manager, which coordinates all agents.
user_proxy.initiate_chat(
    manager,
    message="""
    Find papers on LLM applications that can enhance or augment human learning from arxiv in the last week,
    create a markdown table of different domains.
    """,
)

# Note: The conversation will proceed automatically. You may be prompted to approve the plan;
# press Enter to auto-approve and continue, or type feedback to guide the agents.

Admin (to chat_manager):


    Find papers on LLM applications that can enhance or augment human learning from arxiv in the last week,
    create a markdown table of different domains.
    

--------------------------------------------------------------------------------

Next speaker: Planner



/Users/sourav.banerjee/Documents/Codebases/2. AI ENGINEERING/Autogen_Demystified/.venv/lib/python3.13/site-packages/autogen/oai/groq.py:317: UserWarning: Cost calculation not available for model openai/gpt-oss-120b
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)


Planner (to chat_manager):

**Proposed Plan to Produce a Markdown Table of Recent arXiv Papers on LLM‑Powered Human Learning**

| Step | Who? | Description | Deliverable |
|------|------|-------------|-------------|
| **1. Define Scope & Keywords** | **Scientist** (domain expert) | • Identify the precise research focus: *LLM applications that enhance or augment human learning*.<br>• Compile a list of relevant keywords and synonyms (e.g., “large language model”, “LLM”, “educational AI”, “personalized tutoring”, “knowledge tracing”, “learning analytics”, “human‑in‑the‑loop”, “instructional augmentation”). | A concise keyword list and inclusion/exclusion criteria. |
| **2. Build arXiv Query** | **Engineer** (software developer) | • Write a short script (Python) that uses the arXiv API (or RSS feed) to retrieve all submissions from the last 7 days.<br>• Apply the keyword list from Step 1 to filter results (title, abstract, and categories).<br>• Store the metadata (title, authors, arXiv ID,

ChatResult(chat_id=249257487749328401643222991937757047957, chat_history=[{'content': '\n    Find papers on LLM applications that can enhance or augment human learning from arxiv in the last week,\n    create a markdown table of different domains.\n    ', 'role': 'assistant', 'name': 'Admin'}, {'content': '**Proposed Plan to Produce a Markdown Table of Recent arXiv Papers on LLM‑Powered Human Learning**\n\n| Step | Who? | Description | Deliverable |\n|------|------|-------------|-------------|\n| **1. Define Scope & Keywords** | **Scientist** (domain expert) | • Identify the precise research focus: *LLM applications that enhance or augment human learning*.<br>• Compile a list of relevant keywords and synonyms (e.g., “large language model”, “LLM”, “educational AI”, “personalized tutoring”, “knowledge tracing”, “learning analytics”, “human‑in‑the‑loop”, “instructional augmentation”). | A concise keyword list and inclusion/exclusion criteria. |\n| **2. Build arXiv Query** | **Engineer** (

In [10]:
# Example: Rendering a markdown table in the notebook
# This demonstrates how the final output table would look when rendered.
# The agents will produce similar output, but with real arXiv data.
from IPython.display import Markdown

text = """
| Domain | Title (linked) | Authors | arXiv ID | Submitted (date) | Brief Insight |
|--------|----------------|---------|----------|------------------|---------------|
| **Intelligent Tutoring Systems** | [*LLM‑Tutor: A Large‑Language‑Model Powered Adaptive Tutor for STEM Problem Solving*](https://arxiv.org/abs/2602.00123) | J. Kim, A. Patel, L. Zhou | 2602.00123 | 2026‑02‑07 | Generates step‑by‑step hints and evaluates student solutions in real time, outperforming baseline rule‑based tutors on a physics problem set. |
| **Adaptive Assessment** | [*Dynamic Question Generation with GPT‑4 for Formative Assessment*](https://arxiv.org/abs/2602.00456) | M. Singh, R. Alvarez | 2602.00456 | 2026‑02‑08 | Uses GPT‑4 to create context‑aware multiple‑choice items that adapt difficulty based on a learner’s recent answers. |
| **Knowledge Tracing / Student Modeling** | [*Neural Knowledge Tracing with LLM‑Enhanced Contextual Embeddings*](https://arxiv.org/abs/2602.00987) | S. Lee, H. Wu, P. García | 2602.00987 | 2026‑02‑09 | Augments traditional knowledge‑tracing RNNs with LLM‑derived concept embeddings, yielding a 12 % AUC lift on the ASSISTments dataset. |
| **Curriculum Generation** | [*Curriculum‑GPT: Automated Sequencing of Learning Modules via Large Language Models*](https://arxiv.org/abs/2602.01234) | Y. Chen, D. Patel, K. O’Neil | 2602.01234 | 2026‑02‑10 | Generates personalized learning paths for introductory programming, validated through a user study with 84 undergraduates. |
| **Learning Analytics & Feedback** | [*Explainable Feedback Generation for Writing with LLaMA‑2*](https://arxiv.org/abs/2602.01567) | A. Rossi, N. Gupta, T. Huang | 2602.01567 | 2026‑02‑10 | Produces sentence‑level feedback that aligns with rubric criteria; human raters preferred it over baseline BERT‑based feedback 71 % of the time. |
| **Human‑Computer Interaction (HCI)** | [*Co‑Creative Learning Interfaces: Pairing LLMs with Sketch‑Based Interaction*](https://arxiv.org/abs/2602.01890) | L. Martínez, S. Kim, J. O’Brien | 2602.01890 | 2026‑02‑11 | Introduces a multimodal UI where learners sketch concepts and the LLM expands them into textual explanations, improving retention in a pilot study. |
| **Educational Content Creation** | [*Auto‑Slide: Generating Lecture Slides from Textbooks Using Claude‑3*](https://arxiv.org/abs/2602.02211) | R. Singh, E. Zhao, M. Patel | 2602.02211 | 2026‑02‑11 | Converts textbook chapters into slide decks with speaker notes; evaluation shows 85 % of generated slides meet instructor quality thresholds. |
| **Personalized Learning Assistants** | [*ChatLearn: A Conversational Agent for Self‑Regulated Learning*](https://arxiv.org/abs/2602.02544) | K. Nguyen, P. Bhatia, S. Lee | 2602.02544 | 2026‑02‑12 | A dialogue system that prompts reflection, sets goals, and monitors progress; pilot with 30 high‑school students showed higher metacognitive scores. |
| **Multimodal Learning** | [*Vision‑Language LLMs for Science Experiment Guidance*](https://arxiv.org/abs/2602.02877) | D. Alvarez, H. Kim, V. Patel | 2602.02877 | 2026‑02‑12 | Combines CLIP‑style visual encoders with LLMs to give step‑by‑step instructions for lab experiments, reducing error rates by 23 % in a controlled lab. |
| **Ethics & Pedagogy** | [*Fairness‑Aware Prompting for Educational LLMs*](https://arxiv.org/abs/2602.03102) | M. O’Connor, L. Wang, J. Silva | 2602.03102 | 2026‑02‑12 | Proposes a prompt‑engineering framework that mitigates bias in tutoring dialogues; experiments show reduced gender‑stereotype propagation. |
| **Meta‑Learning for Education** | [*Meta‑LLM: Rapid Adaptation of Large Language Models to New Curriculum Domains*](https://arxiv.org/abs/2602.03455) | S. Gupta, Y. Liu, A. Fernández | 2602.03455 | 2026‑02‑13 | Uses a few‑shot meta‑learning loop to fine‑tune an LLM for a novel subject (e.g., bioinformatics) within 5 minutes, preserving general knowledge. |
"""

# Display the markdown table (renders nicely in Jupyter notebooks)
Markdown(text)


| Domain | Title (linked) | Authors | arXiv ID | Submitted (date) | Brief Insight |
|--------|----------------|---------|----------|------------------|---------------|
| **Intelligent Tutoring Systems** | [*LLM‑Tutor: A Large‑Language‑Model Powered Adaptive Tutor for STEM Problem Solving*](https://arxiv.org/abs/2602.00123) | J. Kim, A. Patel, L. Zhou | 2602.00123 | 2026‑02‑07 | Generates step‑by‑step hints and evaluates student solutions in real time, outperforming baseline rule‑based tutors on a physics problem set. |
| **Adaptive Assessment** | [*Dynamic Question Generation with GPT‑4 for Formative Assessment*](https://arxiv.org/abs/2602.00456) | M. Singh, R. Alvarez | 2602.00456 | 2026‑02‑08 | Uses GPT‑4 to create context‑aware multiple‑choice items that adapt difficulty based on a learner’s recent answers. |
| **Knowledge Tracing / Student Modeling** | [*Neural Knowledge Tracing with LLM‑Enhanced Contextual Embeddings*](https://arxiv.org/abs/2602.00987) | S. Lee, H. Wu, P. García | 2602.00987 | 2026‑02‑09 | Augments traditional knowledge‑tracing RNNs with LLM‑derived concept embeddings, yielding a 12 % AUC lift on the ASSISTments dataset. |
| **Curriculum Generation** | [*Curriculum‑GPT: Automated Sequencing of Learning Modules via Large Language Models*](https://arxiv.org/abs/2602.01234) | Y. Chen, D. Patel, K. O’Neil | 2602.01234 | 2026‑02‑10 | Generates personalized learning paths for introductory programming, validated through a user study with 84 undergraduates. |
| **Learning Analytics & Feedback** | [*Explainable Feedback Generation for Writing with LLaMA‑2*](https://arxiv.org/abs/2602.01567) | A. Rossi, N. Gupta, T. Huang | 2602.01567 | 2026‑02‑10 | Produces sentence‑level feedback that aligns with rubric criteria; human raters preferred it over baseline BERT‑based feedback 71 % of the time. |
| **Human‑Computer Interaction (HCI)** | [*Co‑Creative Learning Interfaces: Pairing LLMs with Sketch‑Based Interaction*](https://arxiv.org/abs/2602.01890) | L. Martínez, S. Kim, J. O’Brien | 2602.01890 | 2026‑02‑11 | Introduces a multimodal UI where learners sketch concepts and the LLM expands them into textual explanations, improving retention in a pilot study. |
| **Educational Content Creation** | [*Auto‑Slide: Generating Lecture Slides from Textbooks Using Claude‑3*](https://arxiv.org/abs/2602.02211) | R. Singh, E. Zhao, M. Patel | 2602.02211 | 2026‑02‑11 | Converts textbook chapters into slide decks with speaker notes; evaluation shows 85 % of generated slides meet instructor quality thresholds. |
| **Personalized Learning Assistants** | [*ChatLearn: A Conversational Agent for Self‑Regulated Learning*](https://arxiv.org/abs/2602.02544) | K. Nguyen, P. Bhatia, S. Lee | 2602.02544 | 2026‑02‑12 | A dialogue system that prompts reflection, sets goals, and monitors progress; pilot with 30 high‑school students showed higher metacognitive scores. |
| **Multimodal Learning** | [*Vision‑Language LLMs for Science Experiment Guidance*](https://arxiv.org/abs/2602.02877) | D. Alvarez, H. Kim, V. Patel | 2602.02877 | 2026‑02‑12 | Combines CLIP‑style visual encoders with LLMs to give step‑by‑step instructions for lab experiments, reducing error rates by 23 % in a controlled lab. |
| **Ethics & Pedagogy** | [*Fairness‑Aware Prompting for Educational LLMs*](https://arxiv.org/abs/2602.03102) | M. O’Connor, L. Wang, J. Silva | 2602.03102 | 2026‑02‑12 | Proposes a prompt‑engineering framework that mitigates bias in tutoring dialogues; experiments show reduced gender‑stereotype propagation. |
| **Meta‑Learning for Education** | [*Meta‑LLM: Rapid Adaptation of Large Language Models to New Curriculum Domains*](https://arxiv.org/abs/2602.03455) | S. Gupta, Y. Liu, A. Fernández | 2602.03455 | 2026‑02‑13 | Uses a few‑shot meta‑learning loop to fine‑tune an LLM for a novel subject (e.g., bioinformatics) within 5 minutes, preserving general knowledge. |
